<a href="https://colab.research.google.com/github/Mufliha-Hafsyah/Machine-Learning/blob/main/Week11/TugasPraktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM PEMBELAJARAN MESIN**

**Nama**  : Mufliha Hafsyah Shahieza <br>
**NIM**   : 244107020147 <br>
**Kelas** : TI-3G

---

# **JS11 - Klasifikasi 2**<hr>
# **Tugas Praktikum**

1. Buatlah model SVM dengan menggunakan data voice.csv dengan ketentuan,<br>
a. Split data dengan rasio 70:30 dan 80:20 untuk setiap model yang akan dibangun.
* Gunakan model dengan kernel linier.
* Gunakan model dengan kernel polynomial.
* Gunakan model dengan kernel RBF.<br>
b. Tabulasikan performansi setiap split dan kernel berdasarkan metrik akurasi.

2. Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!
<hr>

## **Soal 1: Model SVM pada voice.csv**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

df = pd.read_csv('voice.csv')
X = df.drop('label', axis=1)
y = df['label'].map({'female': 0, 'male': 1})

splits = {'70:30': 0.3, '80:20': 0.2}
kernels = ['linear', 'poly', 'rbf']

hasil = []
for nama_split, ts in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=ts, random_state=42, stratify=y)
    for kernel in kernels:
        model = make_pipeline(StandardScaler(), SVC(kernel=kernel))
        model.fit(X_train, y_train)
        hasil.append({
            'Split': nama_split,
            'Kernel': kernel,
            'Akurasi train': accuracy_score(y_train, model.predict(X_train)),
            'Akurasi test': accuracy_score(y_test, model.predict(X_test)),
        })

tabel = pd.DataFrame(hasil)
print(tabel.round(4))

# Tabel ringkas sesuai permintaan soal (akurasi test)
tabel.pivot(index='Kernel', columns='Split', values='Akurasi test').round(4)

   Split  Kernel  Akurasi train  Akurasi test
0  70:30  linear         0.9720        0.9790
1  70:30    poly         0.9666        0.9600
2  70:30     rbf         0.9838        0.9832
3  80:20  linear         0.9767        0.9748
4  80:20    poly         0.9680        0.9558
5  80:20     rbf         0.9838        0.9826


Split,70:30,80:20
Kernel,,
linear,0.9790,0.9748
poly,0.9600,0.9558
rbf,0.9832,0.9826


## **Hasil Soal 1: SVM pada voice.csv**

| Kernel | Split 70:30 | Split 80:20 |
|---|---|---|
| Linear | 0,9790 | 0,9748 |
| Polynomial | 0,9600 | 0,9558 |
| RBF | 0,9832 | 0,9826 |

(akurasi pada data uji; seluruh fitur di-scale dengan StandardScaler, parameter kernel menggunakan nilai bawaan scikit-learn)

Kernel RBF memberikan akurasi tertinggi pada kedua rasio split (0,9832 dan 0,9826), diikuti kernel linear, sedangkan kernel polynomial memiliki akurasi terendah (0,9600 dan 0,9558). Selisih antara RBF dan linear sangat kecil (sekitar 4-5 sampel data uji), sedangkan selisih dengan polynomial lebih jelas. Rendahnya hasil polynomial kemungkinan disebabkan parameter bawaan (derajat 3, C=1) yang kurang sesuai dan belum di-tuning.

Perbedaan rasio split 70:30 dan 80:20 tidak memberikan perubahan berarti (selisih maksimal sekitar 0,4%), sehingga rasio split pada dataset ini tidak menjadi faktor penentu. Selisih akurasi train dan test pada semua model kecil, sehingga tidak ada indikasi overfitting.

## **Soal 2: Citra siang/malam dengan fitur histogram, SVM RBF, 80:20**

In [5]:
!unzip -oq images.zip

In [10]:
from pathlib import Path
import matplotlib.image as mpimg
import cv2
import numpy as np

def load_dataset(img_dir):
    img_list = []
    for folder in Path(img_dir).glob('*'):
        label = folder.name               # 'day' atau 'night'
        for file in folder.glob('*.jpg'):
            img = mpimg.imread(file)
            if img is not None:
                img_list.append((img, label))
    return img_list

# Gabungkan data latih dan uji bawaan dulu, baru kita bagi ulang 80:20
all_img = load_dataset('images/training/') + load_dataset('images/test/')
print("Total gambar:", len(all_img))      # seharusnya 400

Total gambar: 400


In [7]:
# Fitur histogram: 16 bin untuk setiap kanal HSV (total 48 fitur), dinormalisasi
def hist_feature(img, bins=16):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    rentang = [(0, 180), (0, 256), (0, 256)]     # H di OpenCV 0-180, S dan V 0-255
    feats = []
    for ch, (lo, hi) in enumerate(rentang):
        h = cv2.calcHist([hsv], [ch], None, [bins], [lo, hi]).flatten()
        feats.append(h / h.sum())
    return np.concatenate(feats)

X_all = np.array([hist_feature(cv2.resize(img, (1100, 600))) for img, _ in all_img])
y_all = np.array([1 if lbl == 'day' else 0 for _, lbl in all_img])
print(X_all.shape, np.bincount(y_all))      # (400, 48) dan jumlah night, day

(400, 48) [200 200]


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
print(X_train.shape, X_test.shape)

model = SVC(kernel='rbf')       # baseline, parameter bawaan
model.fit(X_train, y_train)
print("Akurasi train:", accuracy_score(y_train, model.predict(X_train)))
print("Akurasi test :", accuracy_score(y_test, model.predict(X_test)))

(320, 48) (80, 48)
Akurasi train: 1.0
Akurasi test : 1.0


In [9]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Sanity check hasil 1,0 pada baseline
print("Akurasi CV tiap fold (seluruh data):", cross_val_score(SVC(kernel='rbf'), X_all, y_all, cv=cv).round(4))
print("Vektor fitur duplikat:", len(X_all) - len(np.unique(X_all.round(8), axis=0)))
print("Akurasi CV baseline (data latih):", round(cross_val_score(SVC(kernel='rbf'), X_train, y_train, cv=cv).mean(), 4))

# Tuning hyperparameter RBF
param_grid = {
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': ['scale', 0.001, 0.01, 0.1, 1, 10]
}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=cv, scoring='accuracy')
grid.fit(X_train, y_train)

print("\nParameter terbaik:", grid.best_params_)
print("Akurasi CV terbaik:", round(grid.best_score_, 4))

hg = pd.DataFrame(grid.cv_results_)
hg['param_gamma'] = hg['param_gamma'].astype(str)
print(hg.pivot(index='param_C', columns='param_gamma', values='mean_test_score').round(4))

# Evaluasi akhir sekali di data uji
best = grid.best_estimator_
print("\nAkurasi train (model terbaik):", accuracy_score(y_train, best.predict(X_train)))
print("Akurasi test (model terbaik) :", accuracy_score(y_test, best.predict(X_test)))

Akurasi CV tiap fold (seluruh data): [1.     1.     1.     0.9875 0.9875]
Vektor fitur duplikat: 1
Akurasi CV baseline (data latih): 0.9906

Parameter terbaik: {'C': 0.1, 'gamma': 'scale'}
Akurasi CV terbaik: 0.9969
param_gamma   0.001    0.01     0.1       1      10   scale
param_C                                                    
0.1          0.9312  0.9281  0.9094  0.9844  0.9656  0.9969
1.0          0.9312  0.9281  0.9781  0.9906  0.9938  0.9906
10.0         0.9312  0.9656  0.9906  0.9969  0.9938  0.9938
100.0        0.9656  0.9906  0.9969  0.9938  0.9938  0.9938
1000.0       0.9906  0.9969  0.9938  0.9938  0.9938  0.9938

Akurasi train (model terbaik): 0.990625
Akurasi test (model terbaik) : 1.0


## **Hasil Soal 2: SVM RBF dengan Fitur Histogram (Citra Siang/Malam)**

Dataset training dan test dari Praktikum 5 digabung (400 gambar: 200 siang dan 200 malam), lalu dibagi ulang dengan rasio 80:20 (320 latih dan 80 uji, stratified). Fitur yang digunakan adalah histogram HSV dengan 16 bin per kanal (48 fitur), dinormalisasi sehingga setiap kanal berjumlah 1.

| Model | Parameter | Akurasi CV | Akurasi Train | Akurasi Test |
|---|---|---|---|---|
| Baseline | C=1, gamma=scale | 0,9906 | 1,0000 | 1,0000 |
| Hasil tuning | C=0,1, gamma=scale | 0,9969 | 0,9906 | 1,0000 |

Tuning dilakukan dengan GridSearchCV (5-fold) pada data latih untuk C ∈ {0,1; 1; 10; 100; 1000} dan gamma ∈ {scale; 0,001; 0,01; 0,1; 1; 10}. Parameter terbaik adalah C=0,1 dan gamma=scale dengan akurasi CV 0,9969. Namun, beberapa kombinasi lain (misalnya C=10 dengan gamma=1, C=100 dengan gamma=0,1, dan C=1000 dengan gamma=0,01) memiliki skor yang sama, sehingga perbedaannya tidak signifikan. Terlihat pola bahwa kombinasi dengan C × gamma ≈ 10 memberikan hasil terbaik, sedangkan kombinasi C dan gamma yang sama-sama kecil menyebabkan underfitting (akurasi sekitar 0,91-0,93).

Akurasi pada data uji mencapai 1,0 baik pada model baseline maupun hasil tuning, sehingga tuning tidak meningkatkan akurasi test. Perlu dicatat bahwa data uji hanya terdiri dari 80 gambar, sehingga akurasi 1,0 tidak berarti model sempurna. Estimasi yang lebih andal dari cross-validation 5-fold pada seluruh data adalah sekitar 0,995. Dibandingkan dengan fitur rata-rata kecerahan pada Praktikum 5 (akurasi test 0,87-0,90), fitur histogram jauh lebih informatif karena memuat sebaran kecerahan dan warna, bukan hanya nilai rata-ratanya. Pemeriksaan duplikat menunjukkan hanya 1 dari 400 vektor fitur yang kembar, sehingga pengaruhnya terhadap hasil evaluasi dapat diabaikan.